*Migrated to NotebookSession API in Phase 4c-i — see notebooks/UTIL_README.md*

# ADP1 BERDL Fitness Flux Fitting

This notebook uses BERDL fitness data (TnSeq-based log2 fold changes) and essentiality data
from minimal media to predict metabolic fluxes in ADP1 on pyruvate media.

**Analysis Pipeline:**
1. Load minimum fitness values (log2 FC) from the BERDL `gene_phenotypes` table
2. Load essentiality data from the `genome_features` table for minimal media
3. Combine into a unified score vector (normalized ratios compatible with `fit_flux_to_mutant_growth_rate_data`)
4. Run unconstrained pFBA on pyruvate media (reference flux)
5. Run fitness-constrained flux fitting using MSExpression
6. Correlate unconstrained vs constrained fluxes and both vs the combined fitness vector

## Load BERDL Fitness Data

Query the `gene_phenotypes` table from `berdl_tables.db` to get the minimum fitness score
(log2 fold change) for each gene across all phenotypes where fitness data exists.

- `fitness_min` is the minimum fitness across replicates for a gene-phenotype pair
- We take `MIN(fitness_min)` across all phenotypes to get the worst-case fitness per gene
- Negative log2FC = reduced fitness when gene is knocked out = gene is important
- Near-zero or positive log2FC = gene knockout has little/no effect = gene is dispensable

In [12]:
from util import session_for
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

session = session_for("ADP1BERDLFitnessFluxFitting.ipynb")

db_path = "data/berdl_tables.db"
conn = sqlite3.connect(db_path)

# Get minimum fitness_min per gene across all phenotypes
query = """
SELECT gene_id,
       MIN(fitness_min) as min_fitness,
       AVG(fitness_avg) as avg_fitness,
       COUNT(*) as phenotype_count
FROM gene_phenotypes
WHERE fitness_match = 'has_score'
  AND fitness_min IS NOT NULL
GROUP BY gene_id
"""

fitness_df = pd.read_sql_query(query, conn)
conn.close()

print(f"Loaded fitness data for {len(fitness_df)} genes")
print(f"\nMinimum fitness (log2 FC) statistics:")
print(fitness_df['min_fitness'].describe())

# Show distribution of fitness values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw log2FC distribution
axes[0].hist(fitness_df['min_fitness'], bins=50, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Minimum Fitness (log2 FC)')
axes[0].set_ylabel('Gene Count')
axes[0].set_title('Distribution of Min Fitness (log2FC) per Gene')
axes[0].axvline(0, color='red', linestyle='--', label='No effect (0)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Converted ratio distribution
fitness_df['ratio'] = (2 ** fitness_df['min_fitness']).clip(upper=1.5)
axes[1].hist(fitness_df['ratio'], bins=50, alpha=0.7, color='coral', edgecolor='black')
axes[1].set_xlabel('Fitness Ratio (2^log2FC, capped at 1.5)')
axes[1].set_ylabel('Gene Count')
axes[1].set_title('Distribution of Converted Fitness Ratios')
axes[1].axvline(1.0, color='red', linestyle='--', label='No effect (1.0)')
axes[1].axvline(0.9, color='green', linestyle='--', label='Activation threshold (0.9)')
axes[1].axvline(0.95, color='orange', linestyle='--', label='Deactivation threshold (0.95)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('nboutput/ADP1BERDLFitnessFluxFitting', exist_ok=True)
plt.savefig('nboutput/ADP1BERDLFitnessFluxFitting/fitness_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Save raw fitness dict (log2FC values)
fitness_dict = dict(zip(fitness_df['gene_id'], fitness_df['min_fitness']))
session.cache.save("berdl_fitness_min_by_gene", fitness_dict)
print(f"\nSaved fitness data for {len(fitness_dict)} genes")

## Load Essentiality Data from Minimal Media

Query the `genome_features` table for the `essentiality_minimal` column, which contains
experimental essentiality calls for minimal media conditions.

- `essential` -> score 0.0 (gene is required, must be active)
- `dispensable` -> score 1.0 (gene is not needed, can be inactive)
- `uncertain` -> score 0.5 (intermediate, treated as mildly important)

This data will overlap minimally with the fitness data, extending our gene coverage.

In [13]:
from util import session_for
import sqlite3
import pandas as pd

session = session_for("ADP1BERDLFitnessFluxFitting.ipynb")

db_path = "data/berdl_tables.db"
conn = sqlite3.connect(db_path)

# Query essentiality for minimal media
query = """
SELECT feature_id as gene_id, essentiality_minimal
FROM genome_features
WHERE essentiality_minimal IS NOT NULL
  AND essentiality_minimal != ''
"""

ess_df = pd.read_sql_query(query, conn)
conn.close()

# Convert to numerical scores
ess_map = {'essential': 0.0, 'dispensable': 1.0, 'uncertain': 0.5}
ess_df['score'] = ess_df['essentiality_minimal'].map(ess_map)

print(f"Loaded essentiality data for {len(ess_df)} genes")
print(f"\nEssentiality distribution:")
print(ess_df['essentiality_minimal'].value_counts())

# Save essentiality scores
ess_dict = dict(zip(ess_df['gene_id'], ess_df['score']))
session.cache.save("berdl_essentiality_minimal_scores", ess_dict)
print(f"\nSaved essentiality scores for {len(ess_dict)} genes")

## Combine Fitness and Essentiality into Unified Score Vector

Create a single coherent array of normalized ratio scores by:
1. Converting log2FC fitness values to ratios: `ratio = 2^(log2FC)`, clamped to [0, 1.5]
2. Merging with essentiality scores
3. For genes with both data sources: use the minimum (most restrictive) score
4. For genes with only one source: use that score

The resulting scores are compatible with `fit_flux_to_mutant_growth_rate_data`:
- Score <= 0.90 (activation_threshold) -> gene classified as "on" (active)
- Score >= 0.95 (deactivation_threshold) -> gene classified as "off" (inactive)

In [14]:
from util import session_for
import numpy as np
import matplotlib.pyplot as plt
import os

session = session_for("ADP1BERDLFitnessFluxFitting.ipynb")

fitness_dict = session.cache.load("berdl_fitness_min_by_gene")
ess_dict = session.cache.load("berdl_essentiality_minimal_scores")

# Convert log2FC fitness to normalized ratio
combined_scores = {}
for gene_id, log2fc in fitness_dict.items():
    ratio = 2 ** log2fc
    ratio = max(0.0, min(ratio, 1.5))  # Clamp to [0, 1.5]
    combined_scores[gene_id] = ratio

print(f"Fitness genes (converted to ratio): {len(combined_scores)}")

# Merge essentiality data
overlap = 0
ess_only = 0
for gene_id, score in ess_dict.items():
    if gene_id in combined_scores:
        # Gene has both: use the more restrictive (lower) score
        combined_scores[gene_id] = min(combined_scores[gene_id], score)
        overlap += 1
    else:
        combined_scores[gene_id] = score
        ess_only += 1

print(f"Overlap (fitness + essentiality): {overlap} genes")
print(f"Essentiality-only genes added: {ess_only}")
print(f"Total combined genes: {len(combined_scores)}")

# Statistics
scores = list(combined_scores.values())
print(f"\nCombined score statistics:")
print(f"  Mean: {np.mean(scores):.4f}")
print(f"  Median: {np.median(scores):.4f}")
print(f"  Min: {np.min(scores):.4f}")
print(f"  Max: {np.max(scores):.4f}")
print(f"  Genes 'on' (score <= 0.90): {sum(1 for s in scores if s <= 0.90)}")
print(f"  Genes 'off' (score >= 0.95): {sum(1 for s in scores if s >= 0.95)}")
print(f"  Genes in between (0.90 < score < 0.95): {sum(1 for s in scores if 0.90 < s < 0.95)}")

# Plot combined distribution
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.hist(scores, bins=50, alpha=0.7, color='mediumpurple', edgecolor='black')
ax.axvline(0.90, color='green', linestyle='--', linewidth=2, label='Activation threshold (0.90)')
ax.axvline(0.95, color='orange', linestyle='--', linewidth=2, label='Deactivation threshold (0.95)')
ax.set_xlabel('Combined Fitness Score (Normalized Ratio)')
ax.set_ylabel('Gene Count')
ax.set_title('Combined Fitness + Essentiality Score Distribution')
ax.legend()
ax.grid(True, alpha=0.3)
os.makedirs('nboutput/ADP1BERDLFitnessFluxFitting', exist_ok=True)
plt.savefig('nboutput/ADP1BERDLFitnessFluxFitting/combined_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Save in format expected by MSExpression.load_from_dict
# Format: {condition_name: {gene_id: value, ...}}
session.cache.save("berdl_combined_fitness_scores", {"combined_fitness": combined_scores})

## Unconstrained pFBA on Pyruvate Media

Run parsimonious FBA (pFBA) on the ADP1 model with pyruvate as the sole carbon source,
without any fitness-based constraints. This provides a reference flux distribution
for comparison with the fitness-constrained solution.

- Model: FullyTranslatedPublishedModel (DgoA knocked out)
- Media: Carbon-D-Glucose base with glucose swapped for pyruvate
- Objective: Maximize growth, then minimize total flux (pFBA)

In [15]:
# TODO (Phase 4d): This cell uses legacy KBase API methods (util.get_media,
# util.constrain_objective_to_fraction_of_optimum) that require the NotebookUtil god-class.
# These methods need KBUtilLib wrappers before this cell can fully migrate.
# For now, session is initialized for cache save/load; legacy util is used for KBase operations.

from util import session_for
import os

session = session_for("ADP1BERDLFitnessFluxFitting.ipynb")

# --- Legacy KBase operations (deferred to Phase 4d) ---
from util_legacy import NotebookUtil
_legacy = NotebookUtil()

from modelseedpy import MSModelUtil, MSMedia
from cobra.flux_analysis import pfba

# Load model and knock out DgoA
model = MSModelUtil.from_cobrapy("models/MergedADP1Model.json")

# Create pyruvate media from Carbon-D-Glucose base
pyruvate_media = _legacy.get_media("KBaseMedia/Carbon-Pyruvic-Acid", msmedia=True)

# Apply media and get optimal growth (fraction=0 means no growth constraint)
optimal_growth = _legacy.constrain_objective_to_fraction_of_optimum(
    model, media=pyruvate_media, objective="MAX{bio1}", fraction=0
)
print(f"Optimal growth on pyruvate: {optimal_growth:.6f}")

# Run pFBA (maximizes growth, then minimizes total flux)
solution = pfba(model.model)
print(f"\npFBA Results:")
print(f"  Status: {solution.status}")
print(f"  Growth rate: {solution.objective_value:.6f}")
active_count = sum(1 for rxn_id in solution.fluxes.index if abs(solution.fluxes[rxn_id]) > 1e-6)
print(f"  Active reactions: {active_count}")

# Save unconstrained fluxes and config via session.cache
unconstrained_fluxes = solution.fluxes.to_dict()
session.cache.save("berdl_unconstrained_pyruvate_fluxes", unconstrained_fluxes)
session.cache.save("berdl_pyruvate_config", {
    "optimal_growth": optimal_growth,
    "media": pyruvate_media.to_dict(output_type="complete")
})
print(f"\nSaved unconstrained fluxes for {len(unconstrained_fluxes)} reactions")

## Fitness-Constrained Flux Fitting

Use `fit_flux_to_mutant_growth_rate_data` from MSExpression to predict flux distributions
that are consistent with the combined fitness/essentiality scores.

The function classifies genes/reactions as "on" or "off" based on thresholds:
- Score <= 0.90 -> gene is important, reaction should be active ("on")
- Score >= 0.95 -> gene is dispensable, reaction can be inactive ("off")

The optimization minimizes flux through "off" reactions while maintaining flux through
"on" reactions, subject to a growth constraint (50% of optimal).

In [16]:
# TODO (Phase 4d): This cell uses legacy KBase API methods (util.constrain_objective_to_fraction_of_optimum,
# util.get_msgenome_from_dict). These need KBUtilLib wrappers before full migration.
# session is used for cache save/load; legacy util handles KBase operations.

from util import session_for
import pandas as pd
import numpy as np

session = session_for("ADP1BERDLFitnessFluxFitting.ipynb")

# Load combined fitness scores and config
expression_data = session.cache.load("berdl_combined_fitness_scores")
config = session.cache.load("berdl_pyruvate_config")

# --- Legacy KBase operations (deferred to Phase 4d) ---
from util_legacy import NotebookUtil
_legacy = NotebookUtil()

from modelseedpy import MSModelUtil, MSMedia, MSExpression

# Create fresh model copy
model = MSModelUtil.from_cobrapy("models/MergedADP1Model.json")
model.model.reactions.get_by_id("DgoA").lower_bound = 0.0
model.model.reactions.get_by_id("DgoA").upper_bound = 0.0
model.model.reactions.get_by_id("rxn01332_c0").lower_bound = -1000
model.model.reactions.get_by_id("rxn01332_c0").upper_bound = 1000

# Apply pyruvate media and constrain growth to 50% of optimal
pyruvate_media = MSMedia.from_dict(config["media"])
constrained_growth = _legacy.constrain_objective_to_fraction_of_optimum(
    model, media=pyruvate_media, objective="MAX{bio1}", fraction=0.5
)
print(f"Growth constrained to >= 50% of optimal: {constrained_growth:.6f}")

# Create MSExpression from combined scores
genome_data = session.cache.load("ADP1Genome")
scores_df = pd.DataFrame(
    list(expression_data["combined_fitness"].items()),
    columns=['gene_id', 'combined_fitness']
)
expression = MSExpression.from_dataframe(
    genome_or_model=_legacy.get_msgenome_from_dict(genome_data["data"]),
    df=scores_df,
    id_column='gene_id',
    type="NormalizedRatios"
)
print(f"MSExpression created with {len(expression._data)} genes")

# Run flux fitting
model.util = _legacy
result = expression.fit_flux_to_mutant_growth_rate_data(
    model=model,
    condition="combined_fitness",
    default_coef=0.01,
    activation_threshold=0.90,
    deactivation_threshold=0.95,
    use_activation_constraints=False
)

print(f"\nFitted Flux Results:")
print(f"  Status: {result['solution'].status}")
if result['solution'].status == 'optimal':
    growth = result['solution'].fluxes.get('bio1', 0)
    print(f"  Growth rate: {growth:.6f}")
    print(f"  On-On reactions (active, expected active): {len(result['on_on'])}")
    print(f"  On-Off reactions (inactive, expected active): {len(result['on_off'])}")
    print(f"  Off-On reactions (active, expected inactive): {len(result['off_on'])}")
    print(f"  Off-Off reactions (inactive, expected inactive): {len(result['off_off'])}")
    print(f"  None-On reactions (active, no data): {len(result['none_on'])}")
    print(f"  None-Off reactions (inactive, no data): {len(result['none_off'])}")
else:
    print(f"  FBA failed: {result['solution'].status}")

# Save constrained result via session.cache
constrained_result = {
    "fluxes": result["solution"].fluxes.to_dict(),
    "growth_rate": result["solution"].fluxes.get("bio1", 0),
    "status": result["solution"].status,
    "on_on": result["on_on"],
    "on_off": result["on_off"],
    "off_on": result["off_on"],
    "off_off": result["off_off"],
    "none_on": result["none_on"],
    "none_off": result["none_off"]
}
session.cache.save("berdl_constrained_pyruvate_result", constrained_result)

# Build reaction-level fitness scores for correlation analysis
rxn_expression = expression.build_reaction_expression(model.model)
rxn_fitness = rxn_expression._data["combined_fitness"].to_dict()
session.cache.save("berdl_reaction_fitness_scores", rxn_fitness)
print(f"\nReaction-level fitness scores computed for {len(rxn_fitness)} reactions")

## Visualize Fitted Flux on Escher Map

Display the fitness-constrained flux solution on the "full" metabolic map using Escher,
matching the visualization approach from ADP1MutantPhenotypeAnalysis.

In [17]:
# TODO (Phase 4d): This cell uses legacy util.create_map_html2() from EscherUtils.
# Migrate to generate_escher_map() when EscherUtils wrappers are available in KBUtilLib.

from util import session_for
import pandas as pd
import os

session = session_for("ADP1BERDLFitnessFluxFitting.ipynb")

# --- Legacy KBase operations (deferred to Phase 4d) ---
from util_legacy import NotebookUtil
_legacy = NotebookUtil()

from modelseedpy import MSModelUtil

# Load the model and fitted flux data
model = MSModelUtil.from_cobrapy("models/MergedADP1Model.json")
constrained_result = session.cache.load("berdl_constrained_pyruvate_result")
fitted_flux = constrained_result["fluxes"]

# Convert to pandas Series for compatibility with create_map_html2
fitted_flux_series = pd.Series(fitted_flux)

output_dir = "nboutput/ADP1BERDLFitnessFluxFitting"
os.makedirs(output_dir, exist_ok=True)
output_path = f"{output_dir}/escher_berdl_fitness_flux.html"

print(f"Growth rate: {constrained_result.get('growth_rate', 'N/A')}")
print(f"Active reactions: {sum(1 for v in fitted_flux.values() if abs(v) > 1e-6)}")
print(f"Output: {output_path}")

output = _legacy.create_map_html2(
    model=model.model,
    flux=fitted_flux_series,
    map="full",
    output_path=output_path,
)
print("Escher map created successfully")

## Evidence Visualization for Active Reactions

For every reaction predicted to carry flux in the constrained solution, show the
essentiality and fitness evidence supporting it:

- **Classification**: on_on (data says active & it is), off_on (data says inactive but model needs it), none_on (no gene-level data)
- **Evidence source per gene**: fitness data only, essentiality only, both, or neither
- **Reaction-level fitness score**: aggregated from gene scores via GPR rules
- **Top reactions by flux**: horizontal bar chart with per-gene annotation

In [18]:
from util import session_for
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

session = session_for("ADP1BERDLFitnessFluxFitting.ipynb")
from matplotlib.patches import Patch
from modelseedpy import MSModelUtil

# Load all required data
model = MSModelUtil.from_cobrapy("models/MergedADP1Model.json")
#model = util.get_model("179225/Abaylyi_ADP1_RASTMS2_OMEGGA_Abaylyi_ADP1_RAST.mdlMS2_OMEGGA_iAbaylyi_Carbon_Succinic.gf")
constrained_result = session.cache.load("berdl_constrained_pyruvate_result")
rxn_fitness = session.cache.load("berdl_reaction_fitness_scores")
fitness_dict = session.cache.load("berdl_fitness_min_by_gene")       # raw log2FC per gene
ess_dict = session.cache.load("berdl_essentiality_minimal_scores")   # essentiality score per gene

fluxes = constrained_result["fluxes"]
on_on  = set(constrained_result["on_on"])
off_on = set(constrained_result["off_on"])
none_on = set(constrained_result["none_on"])

# Identify active reactions (|flux| > 1e-6), exclude exchanges/sinks
active_rxns = [r for r in fluxes if abs(fluxes[r]) > 1e-6
               and not r.startswith("EX_") and not r.startswith("SK_")]
print(f"Active internal reactions: {len(active_rxns)}")

# Build per-reaction evidence table
rows = []
for rxn_id in active_rxns:
    try:
        rxn = model.model.reactions.get_by_id(rxn_id)
        rxn_name = rxn.name or rxn_id
        genes = [str(g) for g in rxn.genes if not str(g).startswith("mRNA_")]
    except KeyError:
        rxn_name = rxn_id
        genes = []

    # Classify reaction
    if rxn_id in on_on:
        classification = "on_on"
    elif rxn_id in off_on:
        classification = "off_on"
    elif rxn_id in none_on:
        classification = "none_on"
    else:
        classification = "other_on"

    # Per-gene evidence
    gene_details = []
    has_fitness = False
    has_ess = False
    for g in genes:
        src = []
        if g in fitness_dict:
            src.append(f"fit={fitness_dict[g]:.2f}")
            has_fitness = True
        if g in ess_dict:
            label = {0.0: "essential", 1.0: "dispensable", 0.5: "uncertain"}.get(ess_dict[g], "?")
            src.append(label)
            has_ess = True
        gene_details.append(f"{g}({'; '.join(src) if src else 'no data'})")

    if has_fitness and has_ess:
        evidence_source = "both"
    elif has_fitness:
        evidence_source = "fitness_only"
    elif has_ess:
        evidence_source = "essentiality_only"
    else:
        evidence_source = "no_data"

    rxn_score = rxn_fitness.get(rxn_id, None)
    rows.append({
        "rxn_id": rxn_id,
        "rxn_name": rxn_name,
        "flux": fluxes[rxn_id],
        "abs_flux": abs(fluxes[rxn_id]),
        "classification": classification,
        "evidence_source": evidence_source,
        "rxn_fitness_score": rxn_score if rxn_score is not None and np.isfinite(rxn_score) else np.nan,
        "n_genes": len(genes),
        "gene_evidence": "; ".join(gene_details) if gene_details else "spontaneous"
    })

df = pd.DataFrame(rows).sort_values("abs_flux", ascending=False)

# ── Colors ──
class_colors = {"on_on": "#2ca02c", "off_on": "#d62728", "none_on": "#7f7f7f", "other_on": "#bcbd22"}
ev_colors = {"both": "#1f77b4", "fitness_only": "#ff7f0e", "essentiality_only": "#9467bd", "no_data": "#d3d3d3"}

# ═══════════════════════════════════════════════════════════════════════
# Figure 1: Overview (3 panels)
# ═══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1 – Counts by classification
class_counts = df["classification"].value_counts()
bars = axes[0].bar(class_counts.index, class_counts.values,
                   color=[class_colors.get(c, "gray") for c in class_counts.index],
                   edgecolor="black")
for bar, v in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 1, str(v), ha="center", fontsize=10)
axes[0].set_ylabel("Reaction count")
axes[0].set_title("Active reactions by classification")
axes[0].grid(axis="y", alpha=0.3)

# Panel 2 – Counts by evidence source
ev_counts = df["evidence_source"].value_counts()
bars2 = axes[1].bar(ev_counts.index, ev_counts.values,
                    color=[ev_colors.get(c, "gray") for c in ev_counts.index],
                    edgecolor="black")
for bar, v in zip(bars2, ev_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 1, str(v), ha="center", fontsize=10)
axes[1].set_ylabel("Reaction count")
axes[1].set_title("Active reactions by evidence source")
axes[1].tick_params(axis="x", rotation=20)
axes[1].grid(axis="y", alpha=0.3)

# Panel 3 – Box plots of fitness score by classification
class_order = [c for c in ["on_on", "off_on", "none_on", "other_on"] if c in df["classification"].values]
box_data = [df.loc[df["classification"] == c, "rxn_fitness_score"].dropna().values for c in class_order]
bp = axes[2].boxplot(box_data, labels=class_order, patch_artist=True, showfliers=True,
                     flierprops=dict(marker=".", markersize=3, alpha=0.4))
for patch, c in zip(bp["boxes"], class_order):
    patch.set_facecolor(class_colors.get(c, "gray"))
    patch.set_alpha(0.6)
axes[2].set_ylabel("Reaction fitness score")
axes[2].set_title("Fitness score distribution\nby classification")
axes[2].axhline(0.90, color="green", linestyle="--", linewidth=1, label="act. thresh (0.90)")
axes[2].axhline(0.95, color="orange", linestyle="--", linewidth=1, label="deact. thresh (0.95)")
axes[2].legend(fontsize=7)
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
os.makedirs("nboutput/ADP1BERDLFitnessFluxFitting", exist_ok=True)
plt.savefig("nboutput/ADP1BERDLFitnessFluxFitting/active_rxn_evidence_overview.png", dpi=150, bbox_inches="tight")
plt.show()

# ═══════════════════════════════════════════════════════════════════════
# Figure 2: Scatter – |flux| vs fitness score, colored by classification
# ═══════════════════════════════════════════════════════════════════════
fig2, ax2 = plt.subplots(figsize=(9, 6))
for cls in ["on_on", "off_on", "none_on", "other_on"]:
    sub = df[df["classification"] == cls].dropna(subset=["rxn_fitness_score"])
    if len(sub) == 0:
        continue
    ax2.scatter(sub["abs_flux"], sub["rxn_fitness_score"], alpha=0.5, s=15,
                color=class_colors[cls], label=f"{cls} ({len(sub)})")
ax2.axhline(0.90, color="green", linestyle="--", linewidth=1)
ax2.axhline(0.95, color="orange", linestyle="--", linewidth=1)
ax2.set_xlabel("|Constrained Flux|")
ax2.set_ylabel("Reaction Fitness Score")
ax2.set_title("Active Reactions: |Flux| vs Fitness Score")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("nboutput/ADP1BERDLFitnessFluxFitting/active_rxn_flux_vs_fitness.png", dpi=150, bbox_inches="tight")
plt.show()

# ═══════════════════════════════════════════════════════════════════════
# Figure 3: Top 40 reactions by |flux| – horizontal bar with gene evidence
# ═══════════════════════════════════════════════════════════════════════
top = df.head(40).copy()
top = top.iloc[::-1]  # reverse so largest flux is at top

fig3, ax3 = plt.subplots(figsize=(12, max(8, len(top) * 0.35)))
bar_colors = [class_colors.get(c, "gray") for c in top["classification"]]
ax3.barh(range(len(top)), top["abs_flux"], color=bar_colors, edgecolor="black", linewidth=0.5)

# Label each bar with rxn name + fitness score + gene evidence summary
for i, (_, row) in enumerate(top.iterrows()):
    score_str = f"{row['rxn_fitness_score']:.2f}" if not np.isnan(row['rxn_fitness_score']) else "N/A"
    label = f"{row['rxn_name']}  [score={score_str}, ev={row['evidence_source']}]"
    ax3.text(top["abs_flux"].max() * 0.01, i, label, va="center", fontsize=7)

ax3.set_yticks(range(len(top)))
ax3.set_yticklabels(top["rxn_id"], fontsize=7)
ax3.set_xlabel("|Constrained Flux|")
ax3.set_title("Top 40 Active Reactions by |Flux| (colored by classification)")
legend_handles = [Patch(facecolor=class_colors[c], edgecolor="black", label=c)
                  for c in ["on_on", "off_on", "none_on"] if c in df["classification"].values]
ax3.legend(handles=legend_handles, fontsize=8, loc="lower right")
ax3.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("nboutput/ADP1BERDLFitnessFluxFitting/top_active_rxn_evidence.png", dpi=150, bbox_inches="tight")
plt.show()

# ═══════════════════════════════════════════════════════════════════════
# Print summary table
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 110)
print(f"{'Rxn ID':<20} {'Class':<10} {'Flux':>10} {'Score':>7} {'Ev.Source':<17} {'Gene Evidence'}")
print("-" * 110)
for _, row in df.head(50).iterrows():
    score_str = f"{row['rxn_fitness_score']:.3f}" if not np.isnan(row['rxn_fitness_score']) else "  N/A"
    gene_str = row['gene_evidence'][:60] + ("..." if len(row['gene_evidence']) > 60 else "")
    print(f"{row['rxn_id']:<20} {row['classification']:<10} {row['flux']:>10.4f} {score_str:>7} "
          f"{row['evidence_source']:<17} {gene_str}")
print("=" * 110)
print(f"\nTotal active internal reactions: {len(df)}")

# Save the full evidence table to TSV
df.to_csv("nboutput/ADP1BERDLFitnessFluxFitting/active_rxn_evidence.tsv", sep="\t", index=False)
print("Full evidence table saved to nboutput/ADP1BERDLFitnessFluxFitting/active_rxn_evidence.tsv")

## Flux-Fitness Correlation Analysis

Correlate three vectors across all reactions:
1. **Unconstrained vs Constrained flux** - How much do fitness constraints change the flux distribution?
2. **|Unconstrained flux| vs Fitness score** - Does unconstrained flux magnitude relate to gene fitness?
3. **|Constrained flux| vs Fitness score** - Does constrained flux magnitude relate to gene fitness?

We use absolute flux values for correlations with fitness since both forward and reverse
flux indicate activity, and we expect active reactions to have lower fitness scores
(their knockouts are more detrimental).

In [19]:
from util import session_for
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy import stats as scipy_stats

session = session_for("ADP1BERDLFitnessFluxFitting.ipynb")

# Load all results
unconstrained_fluxes = session.cache.load("berdl_unconstrained_pyruvate_fluxes")
constrained_result = session.cache.load("berdl_constrained_pyruvate_result")
constrained_fluxes = constrained_result["fluxes"]
rxn_fitness = session.cache.load("berdl_reaction_fitness_scores")

# Find common reactions across all three data sources
common_rxns = set(unconstrained_fluxes.keys()) & set(constrained_fluxes.keys()) & set(rxn_fitness.keys())
# Filter to internal reactions (exclude exchanges/sinks for cleaner correlation)
# Also filter out reactions with NaN or Inf fitness scores
internal_rxns = sorted([
    r for r in common_rxns
    if not r.startswith('EX_') and not r.startswith('SK_')
    and rxn_fitness[r] is not None
    and np.isfinite(rxn_fitness[r])
])

print(f"Reactions in unconstrained: {len(unconstrained_fluxes)}")
print(f"Reactions in constrained: {len(constrained_fluxes)}")
print(f"Reactions with fitness scores: {len(rxn_fitness)}")
nan_count = sum(1 for r in rxn_fitness if rxn_fitness[r] is None or not np.isfinite(rxn_fitness[r]))
print(f"Reactions with NaN/Inf fitness scores (excluded): {nan_count}")
print(f"Common reactions: {len(common_rxns)}")
print(f"Internal reactions for correlation: {len(internal_rxns)}")

# Build paired vectors
unc = np.array([unconstrained_fluxes[r] for r in internal_rxns])
con = np.array([constrained_fluxes[r] for r in internal_rxns])
fit = np.array([rxn_fitness[r] for r in internal_rxns])

# Compute Pearson and Spearman correlations
r1, p1 = scipy_stats.pearsonr(unc, con)
r2, p2 = scipy_stats.pearsonr(np.abs(unc), fit)
r3, p3 = scipy_stats.pearsonr(np.abs(con), fit)
rho1, sp1 = scipy_stats.spearmanr(unc, con)
rho2, sp2 = scipy_stats.spearmanr(np.abs(unc), fit)
rho3, sp3 = scipy_stats.spearmanr(np.abs(con), fit)

print(f"\nCorrelation Results ({len(internal_rxns)} internal reactions):")
print("=" * 75)
print(f"{'Comparison':<42} {'Pearson':>8} {'p-value':>10} {'Spearman':>9}")
print("-" * 75)
print(f"{'Unconstrained vs Constrained flux':<42} {r1:>8.4f} {p1:>10.2e} {rho1:>9.4f}")
print(f"{'|Unconstrained flux| vs Fitness score':<42} {r2:>8.4f} {p2:>10.2e} {rho2:>9.4f}")
print(f"{'|Constrained flux| vs Fitness score':<42} {r3:>8.4f} {p3:>10.2e} {rho3:>9.4f}")
print("=" * 75)

# Scatter plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Unconstrained vs Constrained
axes[0].scatter(unc, con, alpha=0.3, s=8, c='steelblue')
lims = [min(unc.min(), con.min()), max(unc.max(), con.max())]
axes[0].plot(lims, lims, 'r--', alpha=0.5, label='y=x')
axes[0].set_xlabel('Unconstrained Flux')
axes[0].set_ylabel('Constrained Flux')
axes[0].set_title(f'Unconstrained vs Constrained\nPearson r={r1:.4f}, Spearman rho={rho1:.4f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. |Unconstrained| vs Fitness
axes[1].scatter(np.abs(unc), fit, alpha=0.3, s=8, c='coral')
axes[1].set_xlabel('|Unconstrained Flux|')
axes[1].set_ylabel('Reaction Fitness Score')
axes[1].set_title(f'|Unconstrained Flux| vs Fitness\nPearson r={r2:.4f}, Spearman rho={rho2:.4f}')
axes[1].grid(True, alpha=0.3)

# 3. |Constrained| vs Fitness
axes[2].scatter(np.abs(con), fit, alpha=0.3, s=8, c='mediumpurple')
axes[2].set_xlabel('|Constrained Flux|')
axes[2].set_ylabel('Reaction Fitness Score')
axes[2].set_title(f'|Constrained Flux| vs Fitness\nPearson r={r3:.4f}, Spearman rho={rho3:.4f}')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('nboutput/ADP1BERDLFitnessFluxFitting', exist_ok=True)
plt.savefig('nboutput/ADP1BERDLFitnessFluxFitting/flux_fitness_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

# Save correlation results
correlation_results = {
    "n_reactions": len(internal_rxns),
    "unconstrained_vs_constrained": {"pearson": r1, "p_value": float(p1), "spearman": rho1},
    "abs_unconstrained_vs_fitness": {"pearson": r2, "p_value": float(p2), "spearman": rho2},
    "abs_constrained_vs_fitness": {"pearson": r3, "p_value": float(p3), "spearman": rho3}
}
session.cache.save("berdl_flux_fitness_correlations", correlation_results)
print(f"\nCorrelation results saved to datacache")